# STOR 120: Foundations of Data Science
## Lab Quiz Practice Notebook — Classes 14 & 15 (Models, Hypothesis Testing & TVD)

Welcome to the Lab Quiz Practice Notebook! This notebook is designed in Jupyter style to help you prepare for lab quizzes and assessments covering **Class 14 (Models & Categorical Sampling)** and **Class 15 (Decisions & Uncertainty / Hypothesis Testing)**.

---

### Notebook Structure
1. **Module 01: Class 14 — Models & Categorical Sampling**
2. **Module 02: Class 14 — Case Studies (Justin Bieber & *Swain v. Alabama*)**
3. **Module 03: Class 15 — The 4-Step Hypothesis Testing Framework**
4. **Module 04: Class 15 — Total Variation Distance (TVD) Math & Code**
5. **Module 05: Class 15 — P-Values, Decision Cutoffs & Error Rates**

In [3]:
# Run this setup cell first
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

---
## Module 01: Class 14 — Models & Categorical Sampling

### Conceptual Overview
* **What is a Model?** A set of assumptions about data and the chance processes that generated it.
* **Assessment Approach:** 
  1. Define model rules/probabilities under chance.
  2. Simulate sample data according to those chance rules thousands of times.
  3. Compare the real-world observed data to the empirical distribution generated by the simulation.
* **`sample_proportions(sample_size, pop_distribution)`:**
  * Takes `sample_size` (number of draws) and `pop_distribution` (an array/list of category probabilities that sum to `1.0`).
  * Returns an array containing the **proportions** of each category in the drawn sample, matching the exact category order of `pop_distribution`.

### Question 1 (Conceptual — `sample_proportions` Mechanics)
Suppose a bag contains red, blue, and green marbles in proportions `[0.20, 0.50, 0.30]`. You execute the following Python statement:

```python
sim_sample = sample_proportions(500, make_array(0.20, 0.50, 0.30))
```

**Questions:**
1. What is the data type and length of `sim_sample`?
2. What does `sim_sample.item(1)` represent?
3. How would you extract the **count** (number of marbles, not proportion) of blue marbles in the simulated sample?

<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
1. **Data Type & Length:** `sim_sample` is a **NumPy array** of length **3**.
2. **`sim_sample.item(1)`:** Represents the **simulated proportion of blue marbles** (the category at index 1) drawn in this 500-marble sample.
3. **Count Extraction:** Multiply the proportion by the sample size:
   ```python
   blue_count = sim_sample.item(1) * 500
   ```
</details>

In [35]:
sim_sample = sample_proportions(100000, make_array(0.20, 0.50, 0.30))
sim_sample

array([ 0.19823,  0.50289,  0.29888])

### Question 2 (Code Tracing — Categorical Simulation)
You want to simulate rolling a fair 6-sided die 120 times and record how often a `6` appears. Complete the code block below:

In [47]:
# Complete the missing expressions
die_model = make_array(1/6, 5/6) # Index 0 = Rolling a 6, Index 1 = Not a 6
sample_proportions(10000, die_model).item(0) * 120
# sim_rolls = sample_proportions(______, die_model)
# num_sixes = ______

19.92

<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
```python
die_model = make_array(1/6, 5/6)
sim_rolls = sample_proportions(120, die_model)
num_sixes = sim_rolls.item(0) * 120
```
**Explanation:** `sample_proportions(120, die_model)` returns proportions for rolling a 6 (`item(0)`) and not rolling a 6 (`item(1)`). Multiplying `item(0)` by 120 yields the simulated count of 6s.
</details>

---
## Module 02: Class 14 — Case Studies (Justin Bieber & *Swain v. Alabama*)

### Case Study Summary
1. **Justin Bieber Instagram Posts:**
   * **Baseline:** 16.6% of all Instagram posts in Q1 2021 contained video.
   * **Observed Data:** 18.5% of Justin Bieber's 124 posts contained video.
   * **Result:** In 10,000 simulated samples of 124 posts under 16.6% chance, 31% of simulated samples had video proportions $\ge 18.5\%$.
   * **Conclusion:** 18.5% is well within ordinary chance variation. The data is consistent with the 16.6% model.
2. **Swain v. Alabama (1965):**
   * **Baseline:** 26% of eligible county jurors were Black.
   * **Observed Data:** A 100-person jury panel contained only 8 Black men (8%).
   * **Result:** In 100,000 simulated random 100-person panels under 26% chance, **zero** trials produced $\le 8$ Black panelists.
   * **Conclusion:** The probability of getting $\le 8$ Black jurors by pure chance is essentially 0%. We **reject the random selection model** (evidence of selection bias).

### Question 3 (Case Study Logic & Decisions)
Why did researchers conclude that Justin Bieber's 18.5% video rate was consistent with pure chance, while Robert Swain's 8% Black juror panel was evidence of bias? Explain using simulation results and tail probabilities.

<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
* **Justin Bieber:** The observed statistic (18.5%) sat in the **heart/center** of the empirical distribution simulated under the 16.6% chance model. About 31% of random simulations produced 18.5% or higher video posts. Because 31% is a large chance, the observed result is easily explained by ordinary chance variation.
* **Swain v. Alabama:** The observed statistic (8 Black jurors) fell **far out in the extreme tail** of the empirical distribution simulated under the 26% chance model. In 100,000 random trials, 0 simulations yielded 8 or fewer Black jurors ($p < 0.00001$). Because this probability is virtually zero, chance alone cannot reasonably explain the outcome, providing strong evidence against the random selection model.
</details>

---
## Module 03: Class 15 — The 4-Step Hypothesis Testing Framework

### Conceptual Overview
1. **Step 1 — State Hypotheses:**
   * **Null Hypothesis ($H_0$):** A well-defined chance model assuming observed differences are due to **ordinary chance variation alone**. $H_0$ is the hypothesis you simulate under!
   * **Alternative Hypothesis ($H_1$):** A competing view stating that a factor **other than chance** created the observed pattern or difference.
2. **Step 2 — Test Statistic:** A single numerical summary computed from the sample data.
3. **Step 3 — Empirical Distribution under $H_0$:** Simulate the test statistic thousands of times assuming $H_0$ is true.
4. **Step 4 — Conclusion / P-Value:** Compare the observed test statistic to the simulated distribution.

### Question 4 (Hypothesis Formulation)
A university claims that 70% of its students commute to campus. A student group suspects that the true percentage of commuters is **lower** than 70%. They survey a simple random sample of 200 students and find that 120 (60%) commute.

**Tasks:**
1. State the **Null Hypothesis ($H_0$)** in words.
2. State the **Alternative Hypothesis ($H_1$)** in words.
3. What is a suitable **Test Statistic** for this test, and which direction of values favors $H_1$?

<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
1. **Null Hypothesis ($H_0$):** The proportion of commuter students in the university population is equal to 0.70 (70%). Any difference in the sample of 200 students is due to ordinary chance variation.
2. **Alternative Hypothesis ($H_1$):** The proportion of commuter students in the university population is **less than** 0.70 (70%).
3. **Test Statistic:** The **number (or proportion) of commuter students** in a sample of 200.
   * **Direction of Support:** **Small values** (values significantly lower than 140 commuters / 70%) favor the Alternative Hypothesis.
</details>

In [ ]:
# Question 5: Complete the simulation blueprint
# null_dist = make_array(______, ______) # Index 0 = Commuter, Index 1 = Non-commuter
# observed_count = 120
# 
# sim_commuters = make_array()
# 
# for i in np.arange(10000):
#     sim_sample = sample_proportions(______, null_dist)
#     sim_count = ______
#     sim_commuters = np.append(sim_commuters, sim_count)
# 
# p_value = ______

<details>
<summary><b>Click to View Solution & Walkthrough for Question 5</b></summary>

#### Solution:
```python
null_dist = make_array(0.70, 0.30)
observed_count = 120

sim_commuters = make_array()

for i in np.arange(10000):
    sim_sample = sample_proportions(200, null_dist)
    sim_count = sim_sample.item(0) * 200
    sim_commuters = np.append(sim_commuters, sim_count)

p_value = np.count_nonzero(sim_commuters <= observed_count) / 10000
```
**Explanation:** Under $H_0$, each sample of 200 is drawn from `[0.70, 0.30]`. `sim_sample.item(0) * 200` converts the simulated proportion of commuters into a count. Because $H_1$ specifies that the true commuting rate is *lower*, small values favor $H_1$, so we count simulated values `<= observed_count`.
</details>

---
## Module 04: Class 15 — Total Variation Distance (TVD) Math & Code

### Conceptual Overview
When comparing multi-category distributions (3 or more categories, such as pea plant flower colors, jury ethnicity proportions, or shirt sizes), a simple difference in proportions is insufficient. We use **Total Variation Distance (TVD)**:

$$\text{TVD} = \frac{1}{2} \sum_{i=1}^k \left\vert \text{Sample } \%_i - \text{Model } \%_i \right\vert$$

* **Key Properties:**
  * TVD is always between 0 (0%, identical distributions) and 1 (100%, completely disjoint distributions).
  * TVD equals the sum of the **positive differences** between the two distributions.
  * For 2-category distributions, TVD equals the absolute difference in one category.

### Question 6 (Hand Calculation of TVD)
A local high school's student body is 40% Freshmen, 30% Sophomores, 20% Juniors, and 10% Seniors. A club of 100 students has the following distribution: 25% Freshmen, 35% Sophomores, 25% Juniors, and 15% Seniors.

| Category | Model (School %) | Sample (Club %) | Difference | Absolute Difference |
| :--- | :--- | :--- | :--- | :--- |
| Freshmen | 0.40 | 0.25 | -0.15 | 0.15 |
| Sophomores | 0.30 | 0.35 | +0.05 | 0.05 |
| Juniors | 0.20 | 0.25 | +0.05 | 0.05 |
| Seniors | 0.10 | 0.15 | +0.05 | 0.05 |

**Task:** Calculate the Total Variation Distance (TVD) between the high school model distribution and the club sample distribution.

<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
1. **Sum of Absolute Differences:**
   $$\sum \vert \text{Sample} - \text{Model} \vert = 0.15 + 0.05 + 0.05 + 0.05 = 0.30$$
2. **Divide by 2:**
   $$\text{TVD} = \frac{0.30}{2} = 0.15 \quad (\text{or } 15\%)$$
3. **Alternative Method (Sum Positive Differences):**
   $$\text{Positive Diffs} = 0.05 + 0.05 + 0.05 = 0.15$$
   Both methods yield **$\text{TVD} = 0.15$**.
</details>

### Question 7 (Python TVD Function Definition)
Write a Python function `compute_tvd(dist1, dist2)` that takes two NumPy arrays of categorical proportions and returns their Total Variation Distance.

In [ ]:
def compute_tvd(dist1, dist2):
    # Write function body
    return ...


<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
```python
def compute_tvd(dist1, dist2):
    return np.sum(np.abs(dist1 - dist2)) / 2
```
**Explanation:** `np.abs(dist1 - dist2)` computes element-wise absolute differences. `np.sum()` sums them up, and dividing by `2` gives the exact TVD.
</details>

---
## Module 05: Class 15 — P-Values, Decision Cutoffs & Error Rates

### Conceptual Overview
* **P-Value (Observed Significance Level):** The probability, assuming $H_0$ is true, of obtaining a test statistic equal to or more extreme than the observed statistic in the direction of $H_1$.
* **Standard Decision Conventions:**
  * **$p \le 0.05$ (5% cutoff):** Result is "statistically significant" $\rightarrow$ **Reject $H_0$**.
  * **$p \le 0.01$ (1% cutoff):** Result is "highly statistically significant" $\rightarrow$ **Reject $H_0$**.
  * **$p > 0.05$:** Result is consistent with $H_0$ $\rightarrow$ **Fail to reject $H_0$**.
* **Type I Error Rate:** If $H_0$ is true and you use a 5% cutoff ($p \le 0.05$), there is a 5% chance of wrongly rejecting $H_0$.
* **P-Hacking / Data Snooping:** Running multiple hypothesis tests on the same dataset until one happens to give a $p \le 0.05$ by pure chance variation.

### Question 8 (P-Value Interpretation & Cutoffs)
A researcher performs a hypothesis test and obtains a P-value of `0.032` (3.2%). 

**Questions:**
1. What does this P-value mean in plain language regarding $H_0$?
2. At the standard $\alpha = 0.05$ cutoff, what is the formal decision (Reject $H_0$ or Fail to Reject $H_0$)?
3. If $H_0$ is actually true in reality, what was the chance that the researcher would obtain a P-value $\le 0.05$ purely due to random chance?

<details>
<summary><b>Click to View Solution & Walkthrough</b></summary>

#### Solution:
1. **Meaning:** Assuming the Null Hypothesis ($H_0$) is true, there is a 3.2% chance of getting a test statistic as extreme as, or more extreme than, the statistic observed in the sample.
2. **Formal Decision:** Because $0.032 \le 0.05$, the result is **statistically significant**, so we **Reject $H_0$** in favor of $H_1$.
3. **Error Chance:** If $H_0$ is true, there is exactly a **5% chance** of obtaining a P-value $\le 0.05$ by pure chance variation (Type I error rate equal to the cutoff).
</details>

---
## Summary Checklist for Lab Quiz Success
* [x] **`sample_proportions(n, dist)`** returns an array of proportions of length equal to `len(dist)`.
* [x] **$H_0$** is ALWAYS the chance model you simulate under.
* [x] **P-value** measures tail area under $H_0$ in the direction of $H_1$.
* [x] **TVD** $= \frac{1}{2} \sum \vert \text{Sample \%} - \text{Model \%} \vert = \sum \text{Positive Diffs}$.
* [x] **$p \le 0.05$** $\rightarrow$ Reject $H_0$; **$p > 0.05$** $\rightarrow$ Fail to reject $H_0$.

Good luck on your Lab Quiz!